### Comparing 5 random splits of the non-imputed dataset and taking their aggregate results

In [30]:
from random import seed
import numpy as np

seeds = np.random.randint(0, 2**31, 5).tolist()
for i in range(4):
    random_seed = seeds[i]
    print(f"Random seed: {random_seed}")




Random seed: 1700920441
Random seed: 1158856074
Random seed: 428458783
Random seed: 1914051966


In [32]:
import importlib
import feature_engineering as fe_module
import model as model_module
import visualization as viz_module
importlib.reload(fe_module)
importlib.reload(model_module)
importlib.reload(viz_module)
from model import train_best_model
from visualization import plot_feature_importance, plot_confusion_mat, plot_roc, plot_pr_curve, plot_shap_summary
import os
import pandas as pd
import random
import numpy as np

master_seed = 42
random.seed(master_seed)
seeds = [random.randint(0, 2**31 - 1) for _ in range(5)]


files = [
    ("datasets/Dataset_v2/pooled_CN.csv", "CN"),
    ("datasets/Dataset_v2/pooled_MCI_AD.csv", "AD"),
]

# params dict: scalar = fixed value, (lo, hi) = int range, (lo, hi, 'log') = log-scale float range
params = {
    'n_estimators':      (100, 1000),
    'max_depth':         (3, 10),
    'learning_rate':     (0.005, 0.3, 'log'),
    'subsample':         (0.2, 1.0),
    'colsample_bytree':  (0.2, 1.0),
    'colsample_bylevel': (0.2, 1.0),
    'colsample_bynode':  (0.2, 1.0),
    'min_child_weight':  (1, 10),
    'gamma':             (0.0, 5.0),
    'reg_alpha':         (1e-4, 10.0, 'log'),
    'reg_lambda':        (1e-4, 10.0, 'log'),
    'max_delta_step':    0,
}

EXPERIMENT  = "experiment_nonimputedseeds_test"
results_dir = f"experiments/{EXPERIMENT}/grid_results"
charts_dir  = f"experiments/{EXPERIMENT}/charts"
os.makedirs(results_dir, exist_ok=True)
os.makedirs(charts_dir,  exist_ok=True)

exp_bayesian_results = {}
exp_bayesian_models  = {}    # { key: (model, cols, summary) }


# ── Training ──────────────────────────────────────────────────────────────────
for path, prog in files:
    df = pd.read_csv(path)
    base = os.path.splitext(os.path.basename(path))[0]
    for seed in seeds:
        key = f"{base}_{prog}_seed{seed}"
        csv_out = os.path.join(results_dir, f"{key}_cv_scores.csv")
        print(f"\n{'='*60}")
        print(f"=== Optuna search: {key} — {len(df)} samples ===")
        print(f"{'='*60}")

        try:
            model, cols, summary = train_best_model(
                df,
                progression_type=prog,
                params=params,
                csv_path=csv_out,
                save_dir=f"experiments/{EXPERIMENT}",
                n_jobs=10,
                n_trials=1000,
                objective_metric='auc',
                model_base_name=key,          # use key to avoid overwriting
                save_artifacts=True,
                random_state=seed,            # pass the current seed
            )
            exp_bayesian_results[key] = pd.read_csv(csv_out)
            exp_bayesian_models[key]  = (model, cols, summary)
        except Exception as e:
            import traceback
            print(f"Error processing {key}: {e}")
            traceback.print_exc()

# ── Visualizations ────────────────────────────────────────────────────────────
for key, (model, cols, summary) in exp_bayesian_models.items():
    print(f"\n{'='*60}")
    print(f"Visualizations — {key}")
    print(f"{'='*60}")

    # Feature importance
    plot_feature_importance(
        model.feature_importances_,
        cols,
        top_n=20,
        title=f"Top 20 Feature Importances — {key}",
        save_path=os.path.join(charts_dir, f"{key}_feature_importance.png"),
    )

    # Confusion matrix
    plot_confusion_mat(
        summary["y_true"],
        summary["y_pred"],
        title=f"Confusion Matrix — {key}",
        save_path=os.path.join(charts_dir, f"{key}_confusion_matrix.png"),
    )

    # ROC curve
    plot_roc(
        summary["y_true"],
        summary["y_proba"],
        title=f"ROC Curve — {key}",
        save_path=os.path.join(charts_dir, f"{key}_roc_curve.png"),
    )

    # Precision-Recall curve
    plot_pr_curve(
        summary["y_true"],
        summary["y_proba"],
        title=f"Precision-Recall Curve — {key}",
        save_path=os.path.join(charts_dir, f"{key}_pr_curve.png"),
    )

    # SHAP beeswarm (uses stored X_train — no recomputation needed)
    plot_shap_summary(
        model,
        summary["X_train"],
        cols,
        title=f"SHAP Summary — {key}",
        save_path=os.path.join(charts_dir, f"{key}_shap_summary.png"),
    )
aucs = []
for key, (_, _, summary) in exp_bayesian_models.items():
    aucs.append(summary["base_auc"])
print(f"Average ROC‑AUC across seeds: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

/Users/aeg00011/Desktop/AD-Research/AD-Early-Prediction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== Optuna search: pooled_CN_CN_seed478163327 — 12092 samples ===


KeyboardInterrupt: 